# Local time series and power spectra

This notebook introduces Mimir without downloading any data. We will:

1. build a deterministic synthetic light curve;
2. validate and clean it with `TimeSeries`;
3. calculate a `nifty-ls` power spectrum; and
4. check the normalization and recover the injected frequency.

Install the packages used here with:

```bash
python -m pip install "mimir-astro @ git+https://github.com/nielsenmb/Mimir.git" matplotlib
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mimir import TimeSeries, power_spectrum

plt.rcParams["figure.figsize"] = (8, 4)


## Make a synthetic light curve

Times are in days and flux is in parts per million (ppm). The injected
sinusoid has a frequency of 800 μHz and a semi-amplitude of 120 ppm. We also
add a short observing gap, a few flagged samples, and one non-finite value so
that the validation step has something realistic to clean.


In [ ]:
rng = np.random.default_rng(42)

cadence_seconds = 120.0
duration_days = 10.0
time = np.arange(0.0, duration_days, cadence_seconds / 86400.0)

injected_frequency_uhz = 800.0
injected_amplitude_ppm = 120.0
phase = 2 * np.pi * injected_frequency_uhz * 1e-6 * time * 86400.0
flux = injected_amplitude_ppm * np.sin(phase)
flux += rng.normal(0.0, 50.0, size=time.size)
flux_err = np.full(time.size, 50.0)

gap = (time > 4.0) & (time < 4.5)
bad_mask = gap.copy()
bad_mask[rng.choice(time.size, size=12, replace=False)] = True
flux[100] = np.nan


## Validate the samples

`TimeSeries` applies the same mask to time, flux, and uncertainties, drops
non-finite samples by default, sorts the observations, and derives basic
sampling metadata. A `True` value in `bad_mask` means “remove this sample.”


In [ ]:
series = TimeSeries(
    time,
    flux,
    flux_err,
    bad_mask=bad_mask,
    time_unit="d",
    flux_unit="ppm",
)

{
    "input samples": series.input_size,
    "retained samples": series.n_samples,
    "removed samples": series.n_removed,
    "cadence (s)": series.cadence * 86400.0,
    "duration (d)": series.duration,
    "duty cycle": series.duty_cycle,
    "has uncertainties": series.has_uncertainties,
}


In [ ]:
fig, ax = plt.subplots()
ax.plot(series.time, series.flux, ".", ms=1.5)
ax.set(
    xlabel=f"Time [{series.time_unit}]",
    ylabel=f"Relative flux [{series.flux_unit}]",
    title="Validated synthetic light curve",
)
plt.show()


## Calculate the spectrum

`power_spectrum` uses `nifty-ls` directly. An oversampling factor larger than
one refines the plotted grid but does not create additional independent
information—neighbouring oversampled bins are correlated.


In [ ]:
spectrum = power_spectrum(time_series=series, oversampling=4)
peak_index = np.argmax(spectrum.power)
recovered_frequency = spectrum.frequency[peak_index]
recovered_amplitude = spectrum.amplitude[peak_index]

{
    "backend": spectrum.backend,
    "frequency spacing (uHz)": spectrum.frequency_spacing,
    "Nyquist frequency (uHz)": spectrum.nyquist_frequency,
    "recovered frequency (uHz)": recovered_frequency,
    "recovered semi-amplitude (ppm)": recovered_amplitude,
}


In [ ]:
fig, ax = plt.subplots()
selection = (spectrum.frequency > 650.0) & (spectrum.frequency < 950.0)
ax.plot(spectrum.frequency[selection], spectrum.amplitude[selection])
ax.axvline(
    injected_frequency_uhz,
    color="tab:red",
    ls="--",
    label="Injected frequency",
)
ax.set(
    xlabel=f"Frequency [{spectrum.frequency_unit}]",
    ylabel=f"Semi-amplitude [{spectrum.flux_unit}]",
    title="Recovered oscillation",
)
ax.legend()
plt.show()


## Check the normalization

At the default `nyquist_factor=1`, Mimir normalizes the one-sided spectrum so
that its integrated power density equals the variance of the input flux. With
constant uncertainties, the weighted and unweighted variances are identical.


In [ ]:
flux_variance = np.var(series.flux)
integrated_power_density = (
    np.sum(spectrum.power_density) * spectrum.frequency_spacing
)

{
    "flux variance (ppm²)": flux_variance,
    "sum of power (ppm²)": np.sum(spectrum.power),
    "integrated power density (ppm²)": integrated_power_density,
    "relative difference": (
        integrated_power_density / flux_variance - 1.0
    ),
}


For short scripts you may pass arrays directly:

```python
spectrum = power_spectrum(
    time,
    flux,
    flux_err,
    time_unit="d",
    flux_unit="ppm",
    oversampling=4,
)
```

Constructing `TimeSeries` explicitly is useful when several later calculations
should share the same validated samples.
